# [Cloud Evaluation](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/cloud-evaluation#cloud-evaluation-preview-with-azure-ai-projects-sdk)
While Azure AI Evaluation SDK client supports running evaluations locally on your own machine, you might want to delegate the job remotely to the cloud. For example, after you ran local evaluations on small test data to help assess your generative AI application prototypes, now you move into pre-deployment testing and need run evaluations on a large dataset. Cloud evaluation frees you from managing your local compute infrastructure, and enables you to integrate evaluations as tests into your CI/CD pipelines. After deployment, you might want to continuously evaluate your applications for post-deployment monitoring.

In this article, you learn how to run cloud evaluation (preview) in pre-deployment testing on a test dataset. Using the Azure AI Projects SDK, you'll have evaluation results automatically logged into your Azure AI project for better observability. This feature supports all Microsoft curated built-in evaluators and your own custom evaluators which can be located in the Evaluator library and have the same project-scope RBAC

# Variables, Constants and Libraries definition

In [1]:
import os, sys
from pprint import pprint
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv  # requires python-dotenv

if not load_dotenv():
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

TO_BE_EVALUATED_FILE = "./synthetic_dataset_cloud3.jsonl"
FILE_VERSION = "1.0"

foundry_project_endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT")
azure_evaluation_compatible_deployment_name = os.environ["AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential(
    exclude_environment_credential=True
)

if not foundry_project_endpoint or not azure_evaluation_compatible_deployment_name or not credential:
    raise ValueError("Not all variables were properly initialized")

print(f"Foundry project endpoint: {foundry_project_endpoint}")

Foundry project endpoint: https://mm-ai-upskilling-project-resourc.services.ai.azure.com/api/projects/ai-upskilling-project


# Create the Foundry Project Client

In [2]:
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(
        endpoint=foundry_project_endpoint,
        credential=credential,
    ) 

# Helper functions

In [3]:
import json
import time
from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError


def increase_version(version) -> str:
    """Return the next version by incrementing its numeric final part."""
    version_split = version.split(".")
    if len(version_split) > 1:
        version_left = ".".join(version_split[:-1])
        version_right = int(version_split[-1]) + 1
        return f"{version_left}.{version_right}"
    return str(int(version) + 1)


def validate_jsonl(file_path: str) -> None:
    """Validate that every non-empty line is a JSON object."""
    with open(file_path, encoding="utf-8") as jsonl_file:
        for line_number, line in enumerate(jsonl_file, start=1):
            if not line.strip():
                raise ValueError(f"Blank line found in {file_path} at line {line_number}")
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {file_path} at line {line_number}, "
                    f"column {error.colno}: {error.msg}"
                ) from error
            if not isinstance(record, dict):
                raise ValueError(
                    f"Expected a JSON object in {file_path} at line {line_number}"
                )


def upload_dataset(
    project_client,
    file_path: str,
    initial_version: str,
    max_version_attempts: int = 20,
    listing_checks: int = 12,
    listing_check_interval: int = 5,
):
    """Upload a dataset using the first usable version that appears in the project listing."""
    if file_path.lower().endswith(".jsonl"):
        validate_jsonl(file_path)

    file_name = file_path.rsplit("/", 1)[-1].rsplit(".", 1)[0]
    file_version = initial_version

    for _ in range(max_version_attempts):
        version_is_listed = any(
            dataset.name == file_name and dataset.version == file_version
            for dataset in project_client.datasets.list()
        )

        try:
            project_client.datasets.get(name=file_name, version=file_version)
            version_exists = True
        except ResourceNotFoundError:
            version_exists = False

        if version_is_listed:
            print(f"Dataset {file_name} version {file_version} is already listed; trying the next version")
            file_version = increase_version(file_version)
            continue

        if version_exists:
            project_client.datasets.delete(name=file_name, version=file_version)
            print(f"Deletion requested for unlisted dataset {file_name} version {file_version}")
            file_version = increase_version(file_version)
            continue

        try:
            dataset = project_client.datasets.upload_file(
                name=file_name,
                file_path=file_path,
                version=file_version,
            )
        except ResourceExistsError:
            print(f"Dataset {file_name} version {file_version} appeared during upload; trying the next version")
            file_version = increase_version(file_version)
            continue

        for _ in range(listing_checks):
            dataset_is_listed = any(
                listed_dataset.name == dataset.name and listed_dataset.version == dataset.version
                for listed_dataset in project_client.datasets.list()
            )
            if dataset_is_listed:
                return dataset
            time.sleep(listing_check_interval)

        raise RuntimeError(
            f"Dataset {dataset.name} version {dataset.version} was uploaded "
            f"but did not appear in project_client.datasets.list() after "
            f"{listing_checks * listing_check_interval} seconds. No deletion was requested."
        )

    raise RuntimeError(
        f"No usable version found for {file_name} after {max_version_attempts} attempts"
    )

# Uploading evaluation data
We provide two ways to register your data in Azure AI project required for evaluations in the cloud:
- From SDK: Upload new data from your local directory to your Azure AI project in the SDK, and fetch the dataset ID as a result
- Given existing datasets uploaded to your Project...

In [4]:
fileDataset = upload_dataset(
    project_client=project_client,
    file_path=TO_BE_EVALUATED_FILE,
    initial_version=FILE_VERSION,
)

data_id = fileDataset.id

print(f"""\nThe file {fileDataset.name} version {fileDataset.version}
has been successfully uploaded by {fileDataset["systemData"]["createdBy"]} 
at {fileDataset["dataUri"]}.""")

Deletion requested for unlisted dataset synthetic_dataset_cloud3 version 1.0
Deletion requested for unlisted dataset synthetic_dataset_cloud3 version 1.1
Dataset synthetic_dataset_cloud3 version 1.2 is already listed; trying the next version

The file synthetic_dataset_cloud3 version 1.3
has been successfully uploaded by Mauro Minella 
at https://saqtuj635pttdhk.blob.core.windows.net/mm-ai-upsk-d71fd650-a73a-5dd8-8b48-f8c15e9d65c1/synthetic_dataset_cloud3.jsonl.


# Specifying built-in evaluators from Evaluator library

In [5]:
from openai.types.eval_create_params import DataSourceConfigCustom

from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)

from azure.ai.projects.models import TestingCriterionAzureAIEvaluator

## Create the OpenAI evaluation client and dataset schema

In [6]:
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "context": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": ["query", "response", "context", "ground_truth"],
    },
    include_sample_schema=True,
)

## Create the testing criteria

In [7]:
testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="f1_score",
        evaluator_name="builtin.f1_score",
        data_mapping={
            "response": "{{item.response}}",
            "ground_truth": "{{item.ground_truth}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="groundedness",
        evaluator_name="builtin.groundedness",
        initialization_parameters={"model": azure_evaluation_compatible_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="relevance",
        evaluator_name="builtin.relevance",
        initialization_parameters={"model": azure_evaluation_compatible_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="violence",
        evaluator_name="builtin.violence",
        initialization_parameters={"model": azure_evaluation_compatible_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="friendliness",
        evaluator_name="friendliness_evaluator",
        evaluator_version="1",
        initialization_parameters={
            "deployment_name": azure_evaluation_compatible_deployment_name,
            "threshold": 3,
        },
        data_mapping={
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="response_length",
        evaluator_name="response_length_score_evaluator",
        evaluator_version="2",
        initialization_parameters={
            "deployment_name": azure_evaluation_compatible_deployment_name,
            "pass_threshold": 0.5,
        },
        data_mapping={
            "answer": "{{item.response}}",
        },
    ),
]

## Create the evaluation object

In [8]:
openai_client = project_client.get_openai_client()

eval_object = openai_client.evals.create(
    name="Mixed Cloud evaluation 001",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)

print(f"Evaluation created: {eval_object.id}")

Evaluation created: eval_d0663a54d5764ba5937418a44a7614c3


## Create the evaluation run

In [9]:
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name="Cloud evaluation run",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileID(type="file_id", id=data_id),
    ),
)

print(f"Evaluation run created: {eval_run.id}")

Evaluation run created: evalrun_9c33f6cc20b84d54a7d06bad6c123f8e


In [10]:
import time

while True:
    eval_run = openai_client.evals.runs.retrieve(
        run_id=eval_run.id,
        eval_id=eval_object.id,
    )
    if eval_run.status in ("completed", "failed", "canceled"):
        break
    print(f"Evaluation status: {eval_run.status}")
    time.sleep(5)

print(f"Evaluation status: {eval_run.status}")
print(f"Report URL: {eval_run.report_url}")

if eval_run.status == "completed":
    output_items = list(
        openai_client.evals.runs.output_items.list(
            run_id=eval_run.id,
            eval_id=eval_object.id,
        )
    )
    pprint(output_items)

Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: in_progress
Evaluation status: completed
Report URL: https://ai.azure.com/nextgen/r/7KLt2w8MQ1GmNFJ1FJnu6g,ai-upskilling-rg,,mm-ai-upskilling-project-resourc,ai-upskilling-project/build/evaluations/eval_d0663a54d5764ba5937418a44a7614c3/run/evalrun_9c33f6cc20b84d54a7d06bad6c123f8e
[OutputItemListResponse(id='1', created_at=1789244883, datasource_item={'query': 'What is the capital of France?', 'context': 'France is in Europe', 'response': 'Paris is the capital of France.', 'ground_truth': 'Paris'}, datasource_item_id=0, eval_id='eval_d0663a54d5764ba5937418a44a7614c3', obje